# List 3
## Author: Jan Pułtorak

In [ ]:
import torch
import numpy as np
import random
import sklearn
from tqdm import tqdm
import matplotlib.pyplot as plt
import torchvision
from torchvision.transforms import v2

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed);

## Dataset preparation

In [ ]:
train_data = torchvision.datasets.FashionMNIST(root='../data', train=True, download=True)
test_data = torchvision.datasets.FashionMNIST(root='../data', train=False, download=True)

train_data.data.shape, test_data.data.shape, train_data.class_to_idx

In [ ]:
def shuffle(x: torch.Tensor):
    return x[torch.randperm(len(x))]

def prepare_data(dataset: torchvision.datasets.FashionMNIST, positive_class=5):
    X_raw  = dataset.data.float()
    X_raw = X_raw.reshape((X_raw.shape[0], -1))

    # print(X_raw.shape)
    y = dataset.targets

    pos_idx = torch.nonzero(y == positive_class).flatten()
    neg_idx = torch.nonzero(y != positive_class).flatten()


    total_pos = len(pos_idx)
    neg_idx_sampled = shuffle(neg_idx)[:total_pos]

    # print(pos_idx.shape, neg_idx_sampled.shape)
    # print(pos_idx[:10], neg_idx_sampled[:10])

    all_idx = shuffle(torch.cat([pos_idx, neg_idx_sampled]))
    # print(all_idx.shape)

    X = X_raw[all_idx]
    X_norm = X / 255.0

    y = (y[all_idx] == positive_class).float().reshape(-1, 1)
    # print(y.shape)

    return X, X_norm, y



In [ ]:
X_train_raw, X_train_norm, y_train = prepare_data(train_data)
X_test_raw, X_test_norm, y_test = prepare_data(test_data)

X_train_raw.shape

### Explanations
1. Class 'Sandal' was chosen as positive one
2. Training dataset is $(12000, 784)$, test dataset is $(2000, 784)$
3. Number of positive and negative samples: 6000/1000 for train/test

## Task 1

In [ ]:
class LogisticRegression:
    def __init__(self, d):

        scale = 1.0/torch.sqrt(torch.tensor(d))
        self.W = scale * torch.randn((d, 1))
        self.b = scale * torch.randn(1)

    def forward(self, X: torch.Tensor):
        return torch.sigmoid(X @ self.W + self.b)

    def loss(self, y: torch.Tensor, probs: torch.Tensor, eps=1e-8):
        n = len(y)
        return -1/n * torch.sum(y*torch.log(probs+eps) + (1-y)*torch.log(1-probs+eps))

    def update_grad(self, X: torch.tensor, y: torch.Tensor, probs: torch.Tensor, learning_rate):
        n = len(y)
        dW = 1/n * X.T @ (probs - y)
        db = 1/n * torch.sum(probs - y)

        self.W -= learning_rate * dW
        self.b -= learning_rate * db

    def train(self, X_train, y_train, X_test, y_test, learning_rate, epochs=200):
        train_losses, test_losses = [], []
        train_accuracies, test_accuracies = [], []

        for _ in tqdm(range(epochs), desc="Training Logistic Regression model"):
            y_pred_train = self.forward(X_train)
            train_loss = self.loss(y_train, y_pred_train)

            y_pred_test = self.forward(X_test)
            test_loss = self.loss(y_test, y_pred_test)

            self.update_grad(X_train, y_train, y_pred_train, learning_rate)

            accuracy_train = ((y_pred_train >= 0.5) == y_train).float().mean()
            accuracy_test = ((y_pred_test >= 0.5) == y_test).float().mean()

            train_losses.append(train_loss.item())
            test_losses.append(test_loss.item())
            train_accuracies.append(accuracy_train.item())
            test_accuracies.append(accuracy_test.item())

        return train_losses, test_losses, train_accuracies, test_accuracies


In [ ]:
d = X_train_norm.shape[1]
model = LogisticRegression(d)

train_losses, test_losses, train_accuracies, test_accuracies = model.train(X_train_norm, y_train, X_test_norm, y_test, learning_rate=0.2, epochs=1000)
train_accuracies[-1], test_accuracies[-1]

In [ ]:
def plot_loss_and_accuracy(train_losses, test_losses, train_accuracies, test_accuracies):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 5))
    ax1.plot(train_losses, label="Training loss")
    ax1.plot(test_losses, label="Test loss")
    ax1.set(xlabel='Epoch', ylabel='Loss', title="Training and Test loss over Epochs")
    ax1.legend()

    ax2.plot(train_accuracies, label="Training accuracy")
    ax2.plot(test_accuracies, label="Test accuracy")
    ax2.set(xlabel='Epoch', ylabel='Loss', title="Training and Test accuracy over Epochs")
    ax2.legend()

plot_loss_and_accuracy(train_losses, test_losses, train_accuracies, test_accuracies)

### Explanations
1. Class 'Sandal' was chosen as positive one
2. Training dataset is $(12000, 784)$, test dataset is $(2000, 784)$
3. Number of positive and negative samples: 6000/1000 for train/test


### Explanations
1. Logistic regression is suitable because it maps any value from $\mathbb{R}$ into $(0, 1)$ so it can be interpreted as probability, and it has many desired properties such as monotonicity and differentiability.
2. $L = -y\log\hat{y} - (1-y)\log(1-\hat{y})$ and $\frac{\partial L}{\partial z} = \frac{\partial L}{\partial \hat{y}}  \frac{\partial \hat{y}}{\partial z} = \left( -\frac{y}{\hat{y}} + \frac{1-y}{1-\hat{y}} \right) \hat{y}(1-\hat{y}) = \hat{y} - y$
3. Because the model is simple - it's derivatives can be easily derived analytically.

## Task 2

In [ ]:
n = 150
d = X_train_norm.shape[1]
X_batch = X_train_norm[:n]
y_batch = y_train[:n]

W = (torch.randn((d, 1)) * 0.01).requires_grad_(True)
b = (torch.randn(1) * 0.01).requires_grad_(True)

z = X_batch @ W + b
probs = torch.sigmoid(z)
eps = 1e-8
loss = -1/n * torch.sum(y_batch * torch.log(probs + eps) + (1 - y_batch) * torch.log(1 - probs + eps))
loss.backward()

with torch.no_grad():
    dW_manual = 1/n * X_batch.T @ (probs - y_batch)
    db_manual = 1/n * torch.sum(probs - y_batch)

W_diff = torch.norm(dW_manual - W.grad, p=2).item()
max_W_diff = torch.max(dW_manual - W.grad).item()
b_diff = torch.abs(db_manual - b.grad).item()


print(f"Difference in W gradients: {W_diff}")
print(f"Max difference in W gradients: {max_W_diff}")
print(f"Difference in b gradients: {b_diff}")

### Explanations
1. They should match because auto grad implements chain rule, which is what we used for derivation of analytical solution.
2. The mismatch could be due to autograd doing some "uneccessary" operations, which we simplified when deriving analytical solution, and floating point error stacked up.
3. Verifying gradients are calculated correctly can serve as a sanity check.

## Task 3

In [ ]:
class MLP:
    def __init__(self, layer_sizes: list[int]):
        self.weights = []
        self.biases = []

        for i in range(len(layer_sizes) - 1):
            w = (torch.randn(layer_sizes[i], layer_sizes[i+1]) * 0.01).requires_grad_(True)
            b = torch.zeros(1, layer_sizes[i+1]).requires_grad_(True)
            self.weights.append(w)
            self.biases.append(b)

    def forward(self, X: torch.Tensor):
        a = X

        for i in range(len(self.weights) - 1):
            z = a @ self.weights[i] + self.biases[i]
            a = torch.tanh(z)

        z_out = a @ self.weights[-1] + self.biases[-1]
        return torch.sigmoid(z_out)

    def loss(self, y: torch.Tensor, probs: torch.Tensor, eps=1e-8):
        n = len(y)
        return -1/n * torch.sum(y * torch.log(probs + eps) + (1 - y) * torch.log(1 - probs + eps))

    def train(self, X_train, y_train, X_test, y_test, learning_rate, epochs=200):
        train_losses, test_losses = [], []
        train_accuracies, test_accuracies = [], []

        for _ in tqdm(range(epochs), desc=f"Training {len(self.weights)-1}-Hidden-Layer MLP"):
            y_pred_train = self.forward(X_train)
            train_loss = self.loss(y_train, y_pred_train)

            train_loss.backward()

            with torch.no_grad():
                for w, b in zip(self.weights, self.biases):
                    w -= learning_rate * w.grad
                    b -= learning_rate * b.grad

                    w.grad.zero_()
                    b.grad.zero_()

            with torch.no_grad():
                y_pred_test = self.forward(X_test)
                test_loss = self.loss(y_test, y_pred_test)

                accuracy_train = ((y_pred_train >= 0.5) == y_train).float().mean()
                accuracy_test = ((y_pred_test >= 0.5) == y_test).float().mean()

            train_losses.append(train_loss.item())
            test_losses.append(test_loss.item())
            train_accuracies.append(accuracy_train.item())
            test_accuracies.append(accuracy_test.item())

        return train_losses, test_losses, train_accuracies, test_accuracies

In [ ]:
d_in = X_train_norm.shape[1]

mlp1 = MLP(layer_sizes=[d_in, 64, 1])
train_losses_mlp1, test_losses_mlp1, train_accuracies_mlp1, test_accuracies_mlp1  = mlp1.train(
    X_train_norm, y_train, X_test_norm, y_test, learning_rate=0.1, epochs=1000
)
train_accuracies_mlp1[-1], test_accuracies_mlp1[-1]

In [ ]:
plot_loss_and_accuracy(train_losses_mlp1, test_losses_mlp1, train_accuracies_mlp1, test_accuracies_mlp1)

### Explanation

1. Logistic regression is a linear classifier, while MLP with one hidden layer and non-linear approximate any continuous function (Universal approximation theorem).
2. We need non-linear activation function in hidden layer, otherwise the entire network would just simplify to a linear classifier.
3. We need to map logits to probabilities.

## Task 4

In [ ]:
d_in = X_train_norm.shape[1]

mlp2 = MLP(layer_sizes=[d_in, 64, 32, 1])
train_losses_mlp2, test_losses_mlp2, train_accuracies_mlp2, test_accuracies_mlp2  = mlp2.train(
    X_train_norm, y_train, X_test_norm, y_test, learning_rate=0.08, epochs=1000
)
train_accuracies_mlp2[-1], test_accuracies_mlp2[-1]

In [ ]:
plot_loss_and_accuracy(train_losses_mlp2, test_losses_mlp2, train_accuracies_mlp2, test_accuracies_mlp2)

### Explanation

Deeper model did not achieve meaningful improvements. Logistic regression already had accuracy greater than $0.95$, so the problem is too simple for deep networks to make a difference.